# KLEOS Logos v0.0.2 — training (H9)Fine-tunes the text tower of **Ministral 3 14B Reasoning** on**kleos-policy-v0.0.7** with QLoRA, on Kaggle's free **2 x T4**, exactly aspre-registered under H9 in `docs/experiments.md`.**Before you run** (docs/logos.md has the full runbook):1. Settings → Accelerator → **GPU T4 x2**; Settings → **Internet on**.2. Add Input → your private dataset holding `kleos-policy-v0.0.7`.3. Set `COMMIT` in the first code cell to the pushed commit's full sha.4. **Save Version → Save & Run All (Commit).** The run is unattended and stops   at the first failure. ESTIMATED 5–6 hours.**To resume** a run that stopped: add this notebook's previous version **output**as an input, then Save & Run All again. Its checkpoints are copied back andtraining continues from the latest one.Nothing here publishes anything. The adapter lands in this version's output:`outputs/kleos-v007-ministral314breasoning-run1/adapter`.

In [ ]:
import timeSESSION_START = time.time()# Set COMMIT to the full sha of the pushed Kleos-Models commit that pre-registered# H9. A branch name is refused: the run must be reproducible from its manifest.COMMIT = "SET-ME"REPO_URL = "https://github.com/TejasNaik24/Kleos-Models.git"REPO = "/tmp/kleos-models"  # code stays out of the saved outputEXPERIMENT_ID = "kleos-v007-ministral314breasoning-run1"DATASET = "/tmp/kleos-data/kleos-policy-v0.0.7"OUTPUTS = "/kaggle/working/outputs"CONFIG = "configs/training/kleos_logos_v002.yaml"CONFIG_HASH = "d1961583546b5761dd854cfe845838ca91a87ea6189d54be617de21085e8cfa9"TEST_SHA256 = "a4decaaf029b227366846d44018b0b1b669d21f068800a9f3175e15ec4783980"BENCHMARK_SHA256 = "a11ffad75f5147f9d0ddad7bad4bfc073dc730df2b19173ff642774233b4b266"if len(COMMIT) != 40 or any(c not in "0123456789abcdef" for c in COMMIT):    raise SystemExit("Set COMMIT to the 40-character sha of the pushed commit (not 'main').")

In [ ]:
import globimport hashlibimport jsonimport osimport shutilimport subprocessimport sysimport threadingimport timePY = sys.executable# Kaggle ends a GPU session at 12 hours; long commands stop here instead, so this# version still finishes and saves its output for the next one to resume from.SESSION_DEADLINE = SESSION_START + 11.25 * 3600STOPPED = Falsedef run(*args, cwd=None, deadline=False):    """Run a command, streaming its output, and stop the notebook if it fails.    With deadline=True the command is stopped at SESSION_DEADLINE; every later    run() is then skipped, and the version ends normally.    """    global STOPPED    command = " ".join(str(a) for a in args)    if STOPPED:        print("skipped (session deadline reached):", command, flush=True)        return ""    print("$", command, flush=True)    process = subprocess.Popen(        [str(a) for a in args],        cwd=cwd or REPO,        stdout=subprocess.PIPE,        stderr=subprocess.STDOUT,        text=True,    )    lines = []    def pump():        for line in process.stdout:            print(line, end="", flush=True)            lines.append(line)    reader = threading.Thread(target=pump, daemon=True)    reader.start()    while process.poll() is None:        if deadline and time.time() > SESSION_DEADLINE:            print("SESSION DEADLINE: stopping so this version's output is saved. "                  "Attach it as input and Save & Run All again to resume.", flush=True)            process.terminate()            try:                process.wait(timeout=120)            except subprocess.TimeoutExpired:                process.kill()                process.wait()            reader.join(timeout=10)            STOPPED = True            return "".join(lines)        time.sleep(2)    reader.join()    if process.returncode != 0:        raise SystemExit(f"FAILED (exit {process.returncode}): {command}")    return "".join(lines)def sha256(path):    with open(path, "rb") as handle:        return hashlib.sha256(handle.read()).hexdigest()def copy_writable(source, target):    """Copy a tree out of /kaggle/input (read-only) so it can be written and removed."""    shutil.copytree(source, target, copy_function=shutil.copyfile)    for root, _, files in os.walk(target):        os.chmod(root, 0o755)        for name in files:            os.chmod(os.path.join(root, name), 0o644)

## Hardware, code, libraries

In [ ]:
# Two T4s (Settings -> Accelerator -> GPU T4 x2) and room for the 28 GB checkpoint.gpus = subprocess.check_output(    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()print("\n".join(gpus))if len(gpus) != 2 or not all("T4" in g for g in gpus):    raise SystemExit("Need exactly 2 x Tesla T4: Settings -> Accelerator -> GPU T4 x2.")for path in ("/kaggle/working", "/tmp"):    print(f"{path}: {shutil.disk_usage(path).free / 1024**3:.1f} GB free")if shutil.disk_usage("/tmp").free < 30 * 1024**3:    raise SystemExit("/tmp needs 30 GB free for the model weights (HF_HOME).")

In [ ]:
shutil.rmtree(REPO, ignore_errors=True)run("git", "clone", "--quiet", REPO_URL, REPO, cwd="/tmp")run("git", "checkout", "--quiet", COMMIT)print(run("git", "log", "-1", "--format=%H %s").strip())

In [ ]:
# Keeps Kaggle's torch (reinstalling it breaks CUDA), then pins the libraries# Hermes and Logos v0.0.1 ran with. --no-deps stops pip replacing torch.run(PY, "scripts/colab_setup.py")run(PY, "-m", "pip", "install", "-q", "--no-deps", *"transformers==5.16.1 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.50.2 tokenizers==0.23.2".split())

In [ ]:
# Weights on /tmp, never /kaggle/working (saved output, 20 GB). No KLEOS_*# variables: they change config_hash.os.environ["HF_HOME"] = "/tmp/hf_cache"os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"# Line-by-line logs from child processes, kept even if one is stopped.os.environ["PYTHONUNBUFFERED"] = "1"os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"stray = [k for k in os.environ if k.startswith("KLEOS_")]if stray:    raise SystemExit(f"Unset {stray}: KLEOS_* variables change config_hash.")run(PY, "-c", "import torch, transformers, peft, bitsandbytes; "    "print(torch.__version__, torch.cuda.device_count(), transformers.__version__, "    "peft.__version__, bitsandbytes.__version__)")

## Data

In [ ]:
# The private Kaggle dataset holding releases/kleos-policy-v0.0.7 (Add Input),# copied to the path the config hash was computed with, in /tmp, which Kaggle# never saves. Identical copies count once; two different releases stop the run.RELEASE_FILES = ("train.jsonl", "validation.jsonl", "test.jsonl", "manifest.json", "RELEASE.lock")releases = {}for path in sorted(glob.glob("/kaggle/input/**/RELEASE.lock", recursive=True)):    directory = os.path.dirname(path)    if all(os.path.exists(os.path.join(directory, n)) for n in RELEASE_FILES):        digest = tuple(sha256(os.path.join(directory, n)) for n in RELEASE_FILES)        releases.setdefault(digest, directory)if len(releases) != 1:    raise SystemExit(f"Attach exactly one kleos-policy-v0.0.7 release; found {sorted(releases.values())}.")source = next(iter(releases.values()))shutil.rmtree(DATASET, ignore_errors=True)copy_writable(source, DATASET)if sha256(os.path.join(DATASET, "test.jsonl")) != TEST_SHA256:    raise SystemExit("test.jsonl is not the sealed one (sha256 a4decaaf...).")print("dataset:", source, "->", DATASET)

## Resume, config hash, planA previous version's checkpoints are restored if attached. The config hash mustbe the pre-registered one before anything trains.

In [ ]:
run_dir = os.path.join(OUTPUTS, EXPERIMENT_ID)previous = [    p for p in glob.glob(f"/kaggle/input/**/outputs/{EXPERIMENT_ID}", recursive=True)    if glob.glob(os.path.join(p, "checkpoint-*"))]if len(previous) > 1:    raise SystemExit(f"Attach at most one previous version; found {previous}.")if previous and not os.path.exists(run_dir):    copy_writable(previous[0], run_dir)# Finding L-F5: a checkpoint missing optimizer or scheduler state would resume with# a fresh optimizer and a restarted schedule, silently. Set such a checkpoint# aside, visibly; auto-resume then uses the newest complete one.REQUIRED = ("adapter_model.safetensors", "optimizer.pt", "scheduler.pt", "trainer_state.json")for checkpoint in sorted(glob.glob(os.path.join(run_dir, "checkpoint-*")),                         key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True):    missing = [n for n in REQUIRED if not os.path.exists(os.path.join(checkpoint, n))]    if not missing:        break    aside = os.path.join(run_dir, "incomplete-" + os.path.basename(checkpoint))    os.rename(checkpoint, aside)    print(f"L-F5: {os.path.basename(checkpoint)} lacks {missing}; set aside as {aside}.")RESUMING = bool(glob.glob(os.path.join(run_dir, "checkpoint-*")))print("resuming from:", previous[0] if previous else "-", "| checkpoints:",      sorted(os.path.basename(p) for p in glob.glob(os.path.join(run_dir, "checkpoint-*"))))

In [ ]:
printed = run(PY, "-c", "import sys; sys.path.insert(0, 'src'); "    "from kleos_models.config import load_config; "    f"print(load_config({CONFIG!r}, dataset_path={DATASET!r}, output_dir={OUTPUTS!r}).config_hash)")if printed.strip().splitlines()[-1] != CONFIG_HASH:    raise SystemExit("config_hash differs from the one pre-registered under H9; stop.")run(PY, "scripts/plan_run.py", "--config", CONFIG, "--seq-length", "736")

## Smoke run and gateTen steps at the real settings, on scratch space, then the go/no-go gate on thetighter GPU's measured memory. Skipped when resuming a run that already passed it.

In [ ]:
if not RESUMING:    shutil.rmtree("/tmp/logos-v002-smoke", ignore_errors=True)    run(PY, "scripts/train.py", "--config", "configs/training/debug_logos_v002.yaml",        "--dataset", DATASET, "--output-dir", "/tmp/logos-v002-smoke",        "--experiment-id", "kleos-logos-v002-smoke-001", deadline=True)    run(PY, "scripts/check_smoke_gate.py",        "--manifest", "/tmp/logos-v002-smoke/kleos-logos-v002-smoke-001/manifest.json",        "--expect-modules", "280", "--expect-trainable", "60948480", "--min-spare-gb", "0.15")

## Train Logos v0.0.2The pre-registered run. `--resume-from-checkpoint auto` continues from the latestcheckpoint when one was restored above, and starts fresh otherwise.

In [ ]:
run(PY, "scripts/train.py", "--config", CONFIG,    "--dataset", DATASET, "--output-dir", OUTPUTS,    "--experiment-id", EXPERIMENT_ID, "--resume-from-checkpoint", "auto", deadline=True)

In [ ]:
# What this version's output holds. Training loss is not a result; H9 is decided# by the evaluation notebook and compare.py.with open(os.path.join(run_dir, "manifest.json")) as handle:    manifest = json.load(handle)print("config_hash:", manifest.get("config_hash"))metrics = manifest.get("metrics", {})print("recorded metrics:", sorted(metrics))for key in ("memory_probe", "peak_memory_gb", "eval_loss"):    if key in metrics:        print(f"{key}: {metrics[key]}")print(sorted(os.listdir(run_dir)))print(subprocess.check_output(["du", "-sh", run_dir], text=True).strip())if STOPPED:    print("NOT FINISHED: stopped at the session deadline. Add this version's output "          "as an input and Save & Run All again; training resumes from its checkpoints.")